# The experiment costs more than the ticket says

The invoice is the small part. Holding out a fifth of the map for eight weeks means not
treating a fifth of the map for eight weeks, and if the treatment works, that is outcome you
chose not to have. It never appears in a budget line, so it never enters the comparison
against the cheaper design that would have answered the same question.

Sometimes the sign flips: if the prior thinks a treatment is net-negative, withholding it
*pays*, and the "cost" of the experiment is negative before any information arrives. Either
way the number belongs in the decision.

An experiment withholds dose from a holdout and forgoes outcome. `ValuePerOutcome` converts
outcome units to a numeraire and records where that number came from (a ledger line, rule 4).
Discounting uses `w_t = (1 + rate)^(−t)`; `mid_horizon_factor` is the mean weight. The
opportunity cost is **signed**,

    OC = holdout · dose_per_period · n · mid_horizon_factor · (E[ratio] · value − dose_cost),

so a treatment the prior thinks is net-negative has a negative cost of withholding. The net
value of an experiment is `EVSI − opportunity_cost − fixed_cost`.

In [ ]:
import numpy as np

from axiom.core import TimeWindow, Unsupported
from axiom.design import (
    CONCURRENT_EXPERIMENTS, CandidateScore, Collision, CollisionKind, DecisionSpec,
    DesignCandidate, EconomicInputs, ExperimentValue, Occupancy, collisions, exclusions,
    variance_inflation,
    LearningPriority, OpportunityCost, Parameter, ProgramSchedule, Recommendation,
    ScheduledExperiment, SensitivityTable, StudySummary, TreatmentCandidate, ValuePerOutcome,
    discount_weights, elasticity, evaluate_candidate, experiment_value, information_value_of,
    mid_horizon_factor, opportunity_cost, pareto_front, perturb, prior_from_history,
    rank_treatments, recommend, schedule_with_cooldown,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, GOOD, caption, compare, lines, mark_x, mark_y, points

enable();  # every axiom result renders itself from here on

In [ ]:
vpo = ValuePerOutcome(value=3.0, outcome_unit="kg", numeraire="USD", source="contract price, 2026 season")
print(vpo.ledger_line().statement)
print("discount weights (8 periods, 2%):", np.round(discount_weights(8, 0.02), 4))
print("mid-horizon factor:", round(mid_horizon_factor(8, 0.02), 4))

In [ ]:
rng = np.random.default_rng(0)
ratio_draws = rng.lognormal(np.log(1.2), 0.3, size=500)  # outcome units per dose unit
oc: OpportunityCost = opportunity_cost(0.25, 8, dose_per_period=20.0, marginal_value_ratio=ratio_draws,
                                      value_per_outcome=vpo, discount_rate=0.02, dose_unit="L", dose_cost_per_unit=1.0)
print(f"dose withheld {oc.dose_withheld:.1f} L (discounted {oc.dose_withheld_discounted:.1f}); outcome forgone {oc.outcome_forgone:.1f} kg")
print(f"opportunity cost {oc.value:.2f} {oc.numeraire}  (ratio {oc.ratio_mean:.3f} ± {oc.ratio_sd:.3f} from {oc.n_ratio_draws} draws)")
negative = opportunity_cost(0.25, 8, 20.0, 0.2, vpo, 0.02, dose_cost_per_unit=1.0)
print("net-negative treatment -> withholding pays:", round(negative.value, 2))

In [ ]:
decision = DecisionSpec(name="continue_dosing", threshold=1.0, value_per_outcome_unit=20000.0, numeraire="USD")
info = information_value_of(decision, prior_mean=1.2, prior_sd=0.5, experiment_se=0.2)
ev: ExperimentValue = experiment_value(info, oc, fixed_cost=250.0)
print(f"information value {ev.information_value:.2f} − opportunity {ev.opportunity_cost:.2f} − fixed {ev.fixed_cost:.2f} = net {ev.net:.2f} {ev.numeraire}")

In [ ]:
fig = compare(
    ["information the experiment buys (EVSI)", "outcome forgone by holding out", "invoice (fixed cost)", "net value of running it"],
    [ev.information_value, -ev.opportunity_cost, -ev.fixed_cost, ev.net],
    highlight="net value of running it",
    value_fmt="{:,.0f}",
    title="What the experiment is actually worth",
    subtitle=f"decision '{decision.name}', 25% holdout for 8 periods, in {ev.numeraire}",
    x_title=f"value ({ev.numeraire})",
)
caption(fig, "The second bar is the one that is usually missing from the comparison — here "
             "it is 40% of the invoice, and on a treatment that works well or a longer "
             "holdout it dominates it. Its sign is not fixed either: withholding a "
             "net-negative treatment pays.")

## Which treatment to learn about next

Past studies are aged with `decayed_sd` and combined by inverse variance into today's prior.
`rank_treatments` scores each candidate's EIG, EVSI and net value; `recommend` is the greedy
knapsack on net value per unit cost, and returns `Unsupported` when nothing is worth running.

In [ ]:
history = [
    StudySummary(treatment="fertilizer", estimate=1.4, se=0.3, periods_ago=12.0, definition="wald"),
    StudySummary(treatment="fertilizer", estimate=1.0, se=0.4, periods_ago=2.0, definition="wald"),
]
mean, sd = prior_from_history(history, half_life_periods=8.0)
print(f"prior from history: {mean:.3f} ± {sd:.3f}")

In [ ]:
candidates = [
    TreatmentCandidate(name="fertilizer", prior_mean=mean, prior_sd=sd, experiment_se=0.2, decision=decision, opportunity_cost=oc.value, fixed_cost=250.0),
    TreatmentCandidate(name="irrigation", prior_mean=0.8, prior_sd=0.6, experiment_se=0.3, decision=decision, opportunity_cost=100.0, fixed_cost=400.0),
    TreatmentCandidate(name="pruning", prior_mean=0.1, prior_sd=0.1, experiment_se=0.3, decision=decision, opportunity_cost=0.0, fixed_cost=60.0),
]
rows = []
for p in rank_treatments(candidates):
    assert isinstance(p, LearningPriority)
    rows.append([p.rank, p.treatment, f"{p.eig:.3f}", f"{p.evoi:.2f}", f"{p.net_value:.2f}", f"{p.cost:.2f}"])
table(rows, headers=("rank", "treatment", "EIG", "EVSI", "net value", "cost"))
rec = recommend(candidates, budget=600.0)
assert isinstance(rec, Recommendation)
print("selected:", rec.selected, "| total net:", round(rec.total_net_value, 2), "| skipped over budget:", rec.detail["skipped_over_budget"] or "none")
print("nothing worth running ->", type(recommend(candidates[2:])).__name__)

In [ ]:
priorities = rank_treatments(candidates)
fig = compare(
    [f"{p.treatment}  (cost {p.cost:,.0f})" for p in priorities],
    [p.net_value for p in priorities],
    highlight=f"{priorities[0].treatment}  (cost {priorities[0].cost:,.0f})",
    value_fmt="{:,.0f}",
    title="What to learn about next",
    subtitle="net value of running one experiment per treatment, against today's prior for each",
    x_title=f"net value ({decision.numeraire})",
)
caption(fig, "The bottom row is negative: a tight prior nowhere near the decision threshold "
             "means the decision is already made, so the experiment buys nothing and costs "
             "its invoice. recommend() declines to fund it, and returns a typed refusal when "
             "nothing on the slate clears its own cost.")

## Scoring concrete designs, the Pareto front, and a program schedule

A `DesignCandidate` is one way of running the experiment: a registered method, size, horizon,
holdout share, the standard error it would achieve, its cost and its cooldown.
`evaluate_candidate` scores it on one decision; `pareto_front` keeps the non-dominated
candidates on named objectives (leading `-` means minimize); `schedule_with_cooldown` lays them
end to end, best net value first.

In [ ]:
economics = EconomicInputs(value_per_outcome=vpo, dose_per_period=20.0, discount_rate=0.02, dose_unit="L", dose_cost_per_unit=1.0, marginal_value_ratio=1.2)
designs = [
    DesignCandidate(name="small_holdout", method="difference_in_differences", n_units=20, n_periods=6, holdout_fraction=0.2, experiment_se=0.35, cost=200.0, cooldown_periods=2),
    DesignCandidate(name="large_holdout", method="cluster_based_regression", n_units=60, n_periods=8, holdout_fraction=0.4, experiment_se=0.12, cost=600.0, n_clusters=12, cooldown_periods=4),
    DesignCandidate(name="switchback", method="switchback", n_units=20, n_periods=10, holdout_fraction=0.5, experiment_se=0.2, cost=300.0, cooldown_periods=1),
    DesignCandidate(name="ghost", method="ghost", n_units=40, n_periods=6, holdout_fraction=0.3, experiment_se=0.3, cost=450.0),
]
scores: list[CandidateScore] = [evaluate_candidate(c, decision, prior_mean=mean, prior_sd=sd, economics=economics) for c in designs]
table(
    [
        [s.name, f"{s.eig:.3f}", f"{s.evsi:.2f}", f"{s.opportunity_cost:.2f}",
         f"{s.cost:.1f}", f"{s.net_value:.2f}", f"{s.power:.3f}"]
        for s in scores
    ],
    headers=("experiment", "EIG", "EVSI", "opportunity cost", "cost", "net value", "power"),
)

In [ ]:
front = pareto_front(scores, objectives=("net_value", "-cost", "eig"))
print("Pareto front:", [s.name for s in front])

In [ ]:
on_front = {s.name for s in front}
fig = points(
    {
        "on the Pareto front": ([s.cost for s in scores if s.name in on_front],
                                [s.net_value for s in scores if s.name in on_front]),
        "dominated": ([s.cost for s in scores if s.name not in on_front],
                      [s.net_value for s in scores if s.name not in on_front]),
    },
    title="Four ways to run the same experiment",
    subtitle="net value against cost — a dominated design is beaten on every objective at once",
    x_title=f"cost ({decision.numeraire})", y_title=f"net value ({decision.numeraire})",
    height=420,
)
for s_ in scores:
    fig.add_annotation(x=s_.cost, y=s_.net_value, text=s_.name, showarrow=False,
                       yshift=14, font={"size": 11, "color": "#52514e"})
mark_y(fig, 0.0, text="break-even")
caption(fig, "Being expensive is not being dominated: a design is only dropped when another "
             "beats it on cost, net value and information together. That is what keeps a "
             "cheap, uninformative experiment out of the shortlist.")
sched: ProgramSchedule = schedule_with_cooldown(scores, horizon_periods=20)
rows = []
for slot in sched.slots:
    assert isinstance(slot, ScheduledExperiment)
    rows.append([slot.name, f"[{slot.start}, {slot.end})", slot.free_at, f"{slot.net_value:.2f}"])
table(rows, headers=("experiment", "runs", "frees the unit at", "net value"))
print("skipped:", sched.skipped, "| total net:", round(sched.total_net_value, 2))

## Sensitivity

`perturb` re-scores every candidate over a grid of one input and records the winner at each
point; a tipping point is a pair of adjacent grid values across which the winner changes.
`elasticity` is `(Δnet / net) / (Δx / x)` at the base point.

In [ ]:
parameter: Parameter = "value_per_outcome"
sensitivity: SensitivityTable = perturb(designs, decision, mean, sd, economics, parameter, grid=(0.5, 1.0, 2.0, 3.0, 5.0, 8.0))
table(
    [
        [f"{g:.1f}", w, str(tuple(round(v) for v in row))]
        for g, w, row in zip(sensitivity.grid, sensitivity.winners, sensitivity.net_values)
    ],
    headers=("decision value", "winner", "net values"),
)
print("base winner:", sensitivity.base_winner, "| stable:", sensitivity.stable, "| tipping points:", sensitivity.tipping_points)
print("elasticities:", {k: round(v, 3) for k, v in elasticity(sensitivity).items()})

In [ ]:
by_design = {d.name: [row[i] for row in sensitivity.net_values] for i, d in enumerate(designs)}
fig = lines(
    sensitivity.grid, by_design,
    title="Where the recommendation changes its mind",
    subtitle="net value of each design as the value of an outcome unit moves",
    x_title="value per outcome unit", y_title=f"net value ({decision.numeraire})",
)
for low, high in sensitivity.tipping_points:
    mark_x(fig, (low + high) / 2, text="winner changes", color=CRITICAL)
mark_y(fig, 0.0, text="")
caption(fig, "A recommendation that survives the whole grid is a recommendation; one that "
             "flips at a value somebody guessed in a meeting is a coin toss with a decimal "
             "point. The tipping points are where the argument actually needs to happen.")

In [ ]:
se_table = perturb(designs, decision, mean, sd, economics, "experiment_se", grid=(0.5, 0.75, 1.0, 1.5, 2.0))
print(se_table.mode, "-> winners across se multipliers:", se_table.winners)

## The experiment the schedule did not know about

`recommend` above ranked four treatments by what learning about each is worth and picked the
ones that fit the budget. It never asked whether they can run at the same time. For a house
running many experiments per party they usually cannot, and the reason is not the budget —
it is that two designs are about to move the same lever in the same markets in the same weeks.

`no_interference` — "a unit's outcome depends only on its own treatment" — is an assumption of
five of the six methods in `design.methods`, and nothing has ever checked it. Between units it
is hard to check. Between *experiments* it is arithmetic, and the calendar has the information
before either one runs.

In [ ]:
plan = [
    Occupancy(experiment="NW-14", units=("london", "leeds", "bristol"),
              window=TimeWindow(start=0, stop=8), treatments=("price",)),
    Occupancy(experiment="NW-15", units=("leeds", "york"),
              window=TimeWindow(start=4, stop=12), treatments=("banner",)),
    Occupancy(experiment="NW-16", units=("london", "york"),
              window=TimeWindow(start=6, stop=10), treatments=("price",)),
    Occupancy(experiment="NW-17", units=("cardiff",),
              window=TimeWindow(start=0, stop=8), treatments=("banner",)),
    Occupancy(experiment="ROLLOUT", units=("leeds", "cardiff"),
              window=TimeWindow(start=2, stop=6), treatments=("layout",), randomized=False),
]
found = collisions(plan)
table([[c.left, c.right, c.kind, ", ".join(c.units), len(c.periods),
        ", ".join(c.treatments) or "—", c.verdict().status] for c in found],
      headers=("left", "right", "kind", "shared units", "weeks", "shared lever", "verdict"),
      title=f"{len(found)} overlapping pairs out of "
            f"{len(collisions(plan, include_disjoint=True))} scheduled together")

Three answers, not two. **Concurrent** is the interesting one: `NW-14` and `NW-15` share Leeds
for four weeks and move different levers, and both are independently randomized — so neither
estimate is biased. Independent randomizations are orthogonal in expectation. What the overlap
costs is *variance*: the other experiment's effect is sitting in your residual.

**Confounded** is the one to stop. `NW-14` and `NW-16` both move `price` on London in weeks 6
and 7, so neither price effect is separately identified, and no assumption rescues that.
`ROLLOUT` confounds everything it touches for a different reason — it was not independently
randomized, so its assignment may correlate with anything.

In [ ]:
concurrent = next(c for c in found if c.kind == "concurrent")
confounded = next(c for c in found if c.kind == "confounded")
for c in (concurrent, confounded):
    verdict = c.verdict()
    print(f"{c.left} x {c.right}: {verdict.status}")
    print(" ", (verdict.reason or c.reason)[:150])
    print("  assumption:", c.assumption().name, "|", c.assumption().state)
print("\n" + CONCURRENT_EXPERIMENTS.challenged_by)

kinds: list[CollisionKind] = ["disjoint", "concurrent", "confounded"]
print("\nthe ladder:", ", ".join(kinds))

### What a concurrent neighbour costs, in the units power speaks

A neighbour that moves the outcome by `effect` on a `share` of your units adds a Bernoulli
component to your residual, so variance goes from `sd²` to `sd² + share·(1 − share)·effect²`.
That is a number, and it converts straight into the sample size the same design now needs: a
neighbour whose effect is the size of your outcome's own standard deviation, on half your
units, costs a quarter more units to hit the same power.

In [ ]:
SD, EFFECT = 5.0, 5.0
rows = []
for share in (0.0, 0.25, 0.5, 0.75, 1.0):
    factor = variance_inflation(EFFECT, share, SD)
    rows.append([f"{share:.0%}", f"{factor:.4f}", f"{factor ** 0.5:.4f}", f"{factor:.2f}x"])
table(rows, headers=("share of your units", "variance factor", "se factor", "n needed"),
      title=f"a neighbour with effect {EFFECT} against your sd {SD}")
print("The ends are free: if nobody or everybody is in the other experiment, it is a constant,")
print("not noise. The worst case is a fifty-fifty neighbour, which is the usual one.")

### Feeding it back into the pick

`exclusions` turns the confounded pairs into the symmetric map `recommend` takes. A candidate
whose exclusion set already holds a selected treatment is skipped however good its net value
is — two experiments that would move the same lever on the same units in the same weeks do
not become compatible by being worth a lot — and the skipped ones are named rather than
dropped silently.

In [ ]:
forbidden = exclusions(found)
table([[name, ", ".join(others)] for name, others in forbidden.items()],
      headers=("experiment", "may not run beside"), title="confounded pairs only")
print("concurrent pairs are a cost, not a conflict, so they are not in the map by default;")
print("pass kinds=('confounded', 'concurrent') for a program that will not accept the variance.")

# irrigation and fertilizer are both worth running; suppose they share markets and weeks
same_markets = {"irrigation": ["fertilizer"], "fertilizer": ["irrigation"]}
free = recommend(candidates)
constrained = recommend(candidates, exclusions=same_markets)
assert isinstance(free, Recommendation) and isinstance(constrained, Recommendation)
table([["no exclusions", ", ".join(free.selected), f"{free.total_net_value:.2f}", "—"],
       ["they collide", ", ".join(constrained.selected),
        f"{constrained.total_net_value:.2f}", constrained.detail["skipped_excluded"] or "—"]],
      headers=("constraint", "selected", "total net value", "skipped as excluded"))

## What this bought you

The full price of an experiment — invoice, forgone outcome, and its sign — against what it is
worth to the decision waiting on it; a slate of candidate treatments ranked by that number;
the non-dominated designs for the winner; a schedule that respects cooldowns; and the
sensitivity grid that says whether any of it survives a different assumption about what an
outcome is worth.

And, before any of it runs, the pairs on that schedule that share units and weeks — separated
into the ones that cost variance and the ones that cost identification, with the second kind
fed back into the pick as a constraint the net value cannot buy off.